# 🌲 Entrenamiento de Proactive Forest para Clasificación de Especies de Iris

## Descripción del Notebook

Este notebook implementa un entrenamiento **clásico de Machine Learning** (sin aprendizaje federado) utilizando el algoritmo **Proactive Forest** sobre el dataset clásico **Iris** para clasificación de especies de flores.

### Objetivos:
1. Cargar y preprocesar el dataset Iris
2. Dividir los datos en conjuntos de entrenamiento y prueba
3. Entrenar un modelo Proactive Forest
4. Evaluar el modelo mediante métricas de clasificación

### Dataset
**Fuente:** `../../../data/iris.csv`  
**Variable objetivo:** `class` (Iris-setosa, Iris-versicolor, Iris-virginica)

### Características del Dataset:
- **4 features**: sepallength, sepalwidth, petallength, petalwidth
- **3 clases**: Iris-setosa, Iris-versicolor, Iris-virginica
- **150 muestras** en total (50 por clase)

---

## 📦 1. Importación de Librerías

In [ ]:
# ============================================================================
# LIBRERÍAS PARA MANIPULACIÓN DE DATOS
# ============================================================================
import pandas as pd
import numpy as np

# ============================================================================
# LIBRERÍAS PARA VISUALIZACIÓN
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================================
# LIBRERÍAS PARA PREPROCESAMIENTO Y MÉTRICAS
# ============================================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ============================================================================
# MÓDULOS PERSONALIZADOS - PROACTIVE FOREST
# ============================================================================
import sys
sys.path.append('../../../')

from src.domain.model.proactive_forest import ProactiveForest
from src.domain.metrics.forest_evaluator import ForestEvaluator

# ============================================================================
# CONFIGURACIÓN DE VISUALIZACIÓN
# ============================================================================
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

import warnings
warnings.filterwarnings('ignore')

print("✅ Todas las librerías importadas correctamente")

## 📊 2. Carga y Exploración del Dataset

In [ ]:
# ============================================================================
# CARGA DEL DATASET
# ============================================================================
print("=" * 80)
print("📊 CARGA DEL DATASET IRIS")
print("=" * 80)

df = pd.read_csv(
    '../../../data/iris.csv',
    encoding='utf-8'
)

print(f"✅ Dataset cargado exitosamente")
print(f"📊 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")

In [ ]:
# ============================================================================
# VISUALIZACIÓN DE LAS PRIMERAS FILAS
# ============================================================================
print("\n📋 Primeras 10 filas del dataset:")
df.head(10)

In [ ]:
# ============================================================================
# INFORMACIÓN GENERAL DEL DATASET
# ============================================================================
print("=" * 80)
print("📝 INFORMACIÓN DEL DATASET")
print("=" * 80)
df.info()

In [ ]:
# ============================================================================
# ESTADÍSTICAS DESCRIPTIVAS
# ============================================================================
print("=" * 80)
print("📈 ESTADÍSTICAS DESCRIPTIVAS")
print("=" * 80)
df.describe()

## 🎯 3. Análisis de la Variable Objetivo

In [ ]:
# ============================================================================
# DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# ============================================================================
print("=" * 80)
print("🎯 DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (class)")
print("=" * 80)
target_counts = df['class'].value_counts()
target_pct = df['class'].value_counts(normalize=True) * 100

distribution_df = pd.DataFrame({
    'Cantidad': target_counts,
    'Porcentaje (%)': target_pct
})
print(distribution_df)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    x=target_counts.index,
    y=target_counts.values,
    ax=axes[0],
    palette='viridis'
)
axes[0].set_title('Distribución Absoluta de Clases', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Clase', fontsize=12)
axes[0].set_ylabel('Cantidad', fontsize=12)
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=11, fontweight='bold')

axes[1].pie(
    target_counts.values,
    labels=target_counts.index,
    autopct='%1.1f%%',
    colors=sns.color_palette('viridis', len(target_counts)),
    explode=[0.05] * len(target_counts)
)
axes[1].set_title('Distribución Porcentual de Clases', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n✅ Dataset balanceado: {len(target_counts)} clases con {target_counts.iloc[0]} muestras cada una")

## 🔧 4. Preprocesamiento de Datos

In [ ]:
# ============================================================================
# CODIFICACIÓN DE LA VARIABLE OBJETIVO
# ============================================================================
print("=" * 80)
print("🔧 PREPROCESAMIENTO DE DATOS")
print("=" * 80)

label_encoder = LabelEncoder()
df['class_encoded'] = label_encoder.fit_transform(df['class'])

class_names = label_encoder.classes_
print(f"\n✅ Variable objetivo codificada:")
print(f"   Clases: {class_names}")
print(f"   Mapeo: {dict(zip(class_names, range(len(class_names))))}")

In [ ]:
# ============================================================================
# SEPARACIÓN DE FEATURES Y TARGET
# ============================================================================
feature_columns = ['sepallength', 'sepalwidth', 'petallength', 'petalwidth']

X = df[feature_columns]
y = df['class_encoded']

print(f"\n📊 Features (X): {X.shape}")
print(f"🎯 Target (y): {y.shape}")

In [ ]:
# ============================================================================
# DIVISIÓN TRAIN-TEST
# ============================================================================
TEST_SIZE = 0.3
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("=" * 80)
print("📊 DIVISIÓN DE DATOS")
print("=" * 80)
print(f"\n📋 Conjunto de entrenamiento: {X_train.shape}")
print(f"📋 Conjunto de prueba: {X_test.shape}")

## 🌲 5. Configuración y Entrenamiento del Modelo Proactive Forest

In [ ]:
# ============================================================================
# CONFIGURACIÓN DEL MODELO PROACTIVE FOREST
# ============================================================================
N_ESTIMATORS = 100
ALPHA = 0.1
RANDOM_STATE = 42
VERBOSE = True

print("=" * 80)
print("⚙️ CONFIGURACIÓN DEL MODELO")
print("=" * 80)
print(f"🌲 Número de árboles (n_estimators): {N_ESTIMATORS}")
print(f"🎲 Tasa de diversidad (alpha): {ALPHA}")
print(f"🎯 Semilla aleatoria (random_state): {RANDOM_STATE}")
print(f"🏷️  Clases: {class_names}")

In [ ]:
# ============================================================================
# CREACIÓN Y ENTRENAMIENTO DEL MODELO
# ============================================================================
print("=" * 80)
print("🚀 INICIANDO ENTRENAMIENTO DEL PROACTIVE FOREST")
print("=" * 80)
print(f"\n📊 Datos de entrenamiento: {X_train.shape[0]} muestras, {X_train.shape[1]} features")
print(f"⏳ Tiempo estimado: ~10-30 segundos...\n")

model = ProactiveForest(
    n_estimators=N_ESTIMATORS,
    alpha=ALPHA,
    random_state=RANDOM_STATE,
    verbose=VERBOSE,
    class_names=class_names
)

model.fit(X_train.values, y_train)

print("\n" + "=" * 80)
print("✅ ENTRENAMIENTO COMPLETADO EXITOSAMENTE")
print("=" * 80)

In [ ]:
# ============================================================================
# INFORMACIÓN DEL MODELO ENTRENADO
# ============================================================================
trees = model.get_trees()

print("=" * 80)
print("📊 INFORMACIÓN DEL MODELO ENTRENADO")
print("=" * 80)
print(f"\n🌲 Número de árboles en el bosque: {len(trees)}")
print(f"📏 Número de features: {X_train.shape[1]}")
print(f"🏷️  Número de clases: {len(class_names)}")

## 🔮 6. Predicción en el Conjunto de Prueba

In [ ]:
# ============================================================================
# PREDICCIÓN EN EL CONJUNTO DE PRUEBA
# ============================================================================
print("🔮 Realizando predicciones en el conjunto de prueba...")

y_pred = model.predict(X_test.values)

print(f"✅ Predicciones completadas: {len(y_pred)} muestras")
print(f"\n📋 Primeras 10 predicciones vs valores reales:")
print("-" * 60)
for i in range(min(10, len(y_pred))):
    real_cls = label_encoder.inverse_transform([y_test[i]])[0]
    pred_cls = label_encoder.inverse_transform([y_pred[i]])[0]
    match = "✓" if y_test[i] == y_pred[i] else "✗"
    print(f"   {i+1:2d}. Real: {real_cls:20} | Pred: {pred_cls:20} {match}")

## 📈 7. Evaluación del Modelo - Métricas de Clasificación

In [ ]:
# ============================================================================
# CÁLCULO DE MÉTRICAS USANDO FOREST EVALUATOR
# ============================================================================
from src.domain.metrics.forest_evaluator import ForestEvaluator

print("=" * 80)
print("📊 CALCULANDO MÉTRICAS DE EVALUACIÓN")
print("=" * 80)

forest_classifier = model._classifier

report = ForestEvaluator.evaluate(
    forest=forest_classifier,
    X=X_test.values,
    y=y_test,
    class_names=class_names
)

print(f"\n✅ Métricas calculadas exitosamente")

In [ ]:
# ============================================================================
# REPORTE DE MÉTRICAS GLOBALES
# ============================================================================
print("=" * 80)
print("📊 MÉTRICAS GLOBALES DEL MODELO")
print("=" * 80)

print(f"\n🎯 Accuracy (Precisión Global):     {report.accuracy:.4f} ({report.accuracy*100:.2f}%)")
print(f"📈 Macro F1-Score:                   {report.macro_f1:.4f} ({report.macro_f1*100:.2f}%)")
print(f"📊 Macro Precision:                  {report.macro_precision:.4f} ({report.macro_precision*100:.2f}%)")
print(f"📉 Macro Recall:                     {report.macro_recall:.4f} ({report.macro_recall*100:.2f}%)")
print(f"\n🌲 Tamaño del Bosque:                {report.forest_size} árboles")

In [ ]:
# ============================================================================
# MÉTRICAS POR CLASE
# ============================================================================
print("=" * 80)
print("📊 MÉTRICAS POR CLASE")
print("=" * 80)

precision_per_class = precision_score(y_test, y_pred, average=None)
recall_per_class = recall_score(y_test, y_pred, average=None)
f1_per_class = f1_score(y_test, y_pred, average=None)

metrics_df = pd.DataFrame({
    'Precision': precision_per_class,
    'Recall': recall_per_class,
    'F1-Score': f1_per_class
}, index=class_names)

print(f"\n📋 Tabla de métricas por clase:")
print(metrics_df.to_string())

In [ ]:
# ============================================================================
# MATRIZ DE CONFUSIÓN
# ============================================================================
print("=" * 80)
print("📊 MATRIZ DE CONFUSIÓN")
print("=" * 80)

cm = confusion_matrix(y_test, y_pred)
print(f"\n{cm}")

# Visualización
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title('Matriz de Confusión - Dataset Iris', fontsize=14, fontweight='bold')
plt.ylabel('Valor Real', fontsize=12)
plt.xlabel('Valor Predicho', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# REPORTE DE CLASIFICACIÓN COMPLETO
# ============================================================================
print("=" * 80)
print("📋 REPORTE DE CLASIFICACIÓN COMPLETO")
print("=" * 80)
print("\n" + classification_report(y_test, y_pred, target_names=class_names))

## 📝 8. Conclusiones

En este notebook hemos:
1. ✅ Cargado y explorado el dataset Iris
2. ✅ Preprocesado los datos (codificación y división train/test)
3. ✅ Entrenado un modelo Proactive Forest
4. ✅ Evaluado el modelo con múltiples métricas de clasificación

### Resultados Clave:
- **Accuracy**: {report.accuracy:.2%}
- **F1-Score Macro**: {report.macro_f1:.2%}
- **Número de árboles**: {len(trees)}

El modelo Proactive Forest ha demostrado su capacidad para clasificar correctamente las tres especies de Iris.